# Validation-curve comparison

Interactive version of `scripts/plot_val_curves.py`.

Each entry of the YAML config is one **configuration**, backed by one or more
training runs — the seed repetitions that `slurm/classify_param_scan.sh`
produces.  For every entry we

1. load the runs' own `.hydra/config.yaml` to recover the loss weights
   (`finetune.loss_weights.*`) and the augmentation strength `lambda`,
2. read the per-epoch metric from each run's Lightning `metrics.csv`,
3. draw the seed-mean curve with a ±spread band, and mark the best (lowest)
   value reached with a star carrying the spread across seeds as an error bar.

A single-run entry degenerates to the old behaviour: plain curve, plain star,
no band, no error bar.

**Where the code lives.** Everything on the data path — run resolution, metric
reading, seed aggregation — is imported from `scripts/plot_val_curves.py`, so
the script and this notebook cannot drift apart.  Only the labelling and the
drawing stay inline below, where they are worth tweaking by hand.

**The star is not the minimum of the mean curve.** It is the mean over seeds of
the best value *each* seed reached, so it sits slightly below the mean curve
whenever the seeds bottom out at different epochs — that is the quantity the
error bar belongs to.

**Colour scale.** `SCALE = 'uniform'` spreads the colours evenly over the
entries ordered by λ, so consecutive λ values are always equally
distinguishable regardless of how they are spaced.  `SCALE = 'value'` instead
maps the colour to the λ *value* through `color.vmin` / `color.vmax`, which
makes the colour quantitative but crowds together λ values that happen to sit
close.  Entries with jet-contrastive weight 0 are drawn in the baseline colour
(black) either way.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from omegaconf import DictConfig, OmegaConf

%matplotlib inline

# Repo root, whether the kernel started in notebooks/ or at the repo root.
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()
)

# The data path (run resolution, metric reading, seed aggregation) is shared
# with the script rather than copied here — that duplication is what let the two
# drift apart before.
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
from plot_val_curves import Entry, load_entry, summarise  # noqa: E402

CONFIG_PATH = REPO_ROOT / 'scripts/configs/val_loss_ce_lambda_scan_reduced.yaml'
SCALE = 'uniform'  # 'uniform' (evenly spaced by rank) | 'value' (proportional to lambda)

## Labels

`Entry`, `load_entry` and `summarise` come from `scripts/plot_val_curves.py`.
An entry names its runs with `dir:` (one run), `cell:` (every `<cell>/seed_*/`
under it) or `dirs:` (an explicit list); `seed_status:` in the config filters
out seeds whose `run_info.yaml` says `failed` or `truncated`, and `error:`
picks `std` / `sem` / `minmax` for the band and the error bar.

Only the legend label stays here, since it is the thing worth rewording per
figure.

In [ ]:
def auto_label(entry: Entry, uniform_jc: bool, show_n: bool = True) -> str:
    """Legend label: only the fields that actually vary across the comparison."""
    if entry.label is not None:
        label = entry.label
    elif entry.jc == 0.0:
        #label = f'no JC  ($\\lambda$ = {entry.lam:g})'
        label = f'no Contr.'
    elif uniform_jc:
        label = f'$\\lambda$ = {entry.lam:g}'
    else:
        label = f'JC {entry.jc:g},  $\\lambda$ = {entry.lam:g}'

    # n is worth showing only where there is something to average over.
    if show_n and entry.n_seeds > 1:
        label += f'  (n = {entry.n_seeds})'
    return label

## Load the configurations

In [ ]:
cfg = OmegaConf.load(CONFIG_PATH)
metric = cfg.get('metric', 'val/head/loss_ce')
error = cfg.get('error', 'std')

entries = [load_entry(raw, metric, cfg) for raw in cfg['models']]

# best_mean ± best_<error> is the star and its bar; common_epochs is how far the
# mean curve extends (the shortest seed's reach), best_per_seed the raw numbers.
summary = summarise(entries, error)
summary

## Colours and plot

In [ ]:
def assign_colors(
    entries: list[Entry], cmap, scale: str, norm: Normalize, baseline_color: str
) -> list:
    """One colour per entry, in the order given.

    ``scale='uniform'`` spaces the scan entries evenly over the colormap by
    their rank in lambda; ``scale='value'`` maps the lambda value through
    ``norm``.  Entries with ``jc == 0`` get ``baseline_color``, and an explicit
    ``color:`` in the config always wins.
    """
    scan = [e for e in entries if e.jc != 0.0]
    rank = {id(e): k for k, e in enumerate(sorted(scan, key=lambda e: e.lam))}
    span = max(len(scan) - 1, 1)

    colors = []
    for entry in entries:
        if entry.color is not None:
            colors.append(entry.color)
        elif entry.jc == 0.0:
            colors.append(baseline_color)
        elif scale == 'uniform':
            colors.append(cmap(rank[id(entry)] / span))
        else:
            colors.append(cmap(norm(entry.lam)))
    return colors


def plot_curves(entries: list[Entry], cfg: DictConfig, scale: str = SCALE):
    """Draw all curves on one axes and return ``(fig, ax)``."""
    style = cfg.get('color', {})
    cmap = LinearSegmentedColormap.from_list(
        'blue_red', list(style.get('cmap', ['#1f3ecc', '#7a1fa2', '#cc1f1f']))
    )
    norm = Normalize(
        vmin=float(style.get('vmin', 0.0)), vmax=float(style.get('vmax', 1.0))
    )

    # Baselines first, then the scan in ascending lambda, so the legend reads as
    # a gradient.
    entries = sorted(entries, key=lambda e: (e.jc != 0.0, e.lam))
    colors = assign_colors(
        entries, cmap, scale, norm, style.get('baseline_color', 'black')
    )
    uniform_jc = len({e.jc for e in entries if e.jc != 0.0}) <= 1

    plot_cfg = cfg.get('plot', {})
    show_n = plot_cfg.get('show_n', True)
    band = plot_cfg.get('band', True)

    fig, ax = plt.subplots(figsize=tuple(plot_cfg.get('figsize', [8.0, 5.0])))

    for entry, color in zip(entries, colors):
        curve = entry.curve

        # Band and mean run only over the epochs EVERY seed reached, so no point
        # of the mean is built from a different set of runs than its neighbours.
        if band and entry.n_seeds > 1:
            ax.fill_between(
                curve['epoch'], curve['lo'], curve['hi'],
                color=color, alpha=plot_cfg.get('band_alpha', 0.15), linewidth=0,
            )

        ax.plot(
            curve['epoch'],
            curve['value'],
            color=color,
            linewidth=1.4,
            label=auto_label(entry, uniform_jc, show_n),
        )
        # The star is the mean over seeds of the best value EACH one reached —
        # not the minimum of the mean curve, which would be biased low by
        # whichever seed happened to dip at that epoch.
        ax.errorbar(
            entry.best_epoch,
            entry.best_value,
            yerr=entry.best_error or None,
            marker='*',
            markersize=12,
            color=color,
            markeredgecolor='white',
            markeredgewidth=0.5,
            linestyle='none',
            elinewidth=1.2,
            capsize=3,
            zorder=5,
        )

    ax.set_xlabel(plot_cfg.get('xlabel', 'Epoch'))
    ax.set_ylabel(plot_cfg.get('ylabel', cfg['metric']))
    if plot_cfg.get('title'):
        ax.set_title(plot_cfg['title'])
    if plot_cfg.get('ylim'):
        ax.set_ylim(*plot_cfg['ylim'])
    if plot_cfg.get('xlim'):
        ax.set_xlim(*plot_cfg['xlim'])
    ax.grid(True, alpha=0.3)

    legend_kwargs = {'fontsize': 8, 'title': plot_cfg.get('legend_title')}
    if plot_cfg.get('legend_outside', True):
        legend_kwargs |= {'loc': 'upper left', 'bbox_to_anchor': (1.01, 1.0)}
    else:
        legend_kwargs |= {'loc': 'best', 'ncol': 2}
    ax.legend(**legend_kwargs)

    # A colourbar only carries meaning when the colour tracks the lambda value.
    if style.get('colorbar', False) and scale == 'value':
        bar = fig.colorbar(
            plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, pad=0.02
        )
        bar.set_label('$\\lambda$')

    fig.tight_layout()
    return fig, ax

In [ ]:
fig, ax = plot_curves(entries, cfg)

## Tweak and save

Re-run the cell above after editing anything — `ax` is live, so you can also
adjust it in place (`ax.set_ylim(...)`, `ax.set_title(...)`) and redisplay
`fig`.  Set `SCALE = 'value'` in the setup cell to colour by the λ value
instead of by rank, `plot.band: false` in the config to drop the bands, and
`error: minmax` to show the full seed range instead of the standard deviation.

In [48]:
out_dir = REPO_ROOT / cfg.get('plot', {}).get('out_dir', 'plots/val_curves')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / cfg.get('plot', {}).get('filename', 'val_curves.png')

fig.savefig(out_path, dpi=200, bbox_inches='tight')
print(f'Saved {out_path}')

Saved /storage3/DSIP/rriva/research/fm4tag/plots/val_loss_ce/val_loss_ce_lambda_scan_5.png
